# SQL Joins

## Overview

This notebook explores SQL JOIN operations using the movie database.

The objective is to understand how relational tables can be combined to produce analytical datasets while maintaining data integrity and controlling row multiplication.

The analysis focuses on:

- Understanding table relationships.
- Applying different SQL JOIN types.
- Understanding table grain.
- Working with one-to-one, one-to-many, and many-to-many relationships.
- Identifying row multiplication.
- Detecting duplicate records introduced by joins.
- Preventing join explosion.
- Aggregating bridge tables before joining.
- Validating join results.
- Applying SQL join optimization techniques.

The notebook uses SQLAlchemy to connect Python to the database and Pandas to retrieve and inspect query results.

## Database

The database contains the following tables:

- actors
- casts
- oscars
- movies
- keywords
- languages
- genres
- genremap
- keywordmap
- languagemap
- productioncompanymap
- productioncountrymap
- productioncountries
- productioncompanies

## Key Principle

A JOIN should always be designed with an understanding of the grain of both tables.

Before joining two tables, we should answer:

> What does one row represent in each table?

Understanding grain is essential for preventing duplicate records and unintended row multiplication.

In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Importing the required libraries
import pandas as pd
from  sqlalchemy import create_engine
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [9]:
# Creating a database engine
db_path = Path.home() / "Documents" / "Movie DB SQL Analytics" / "Database" / "TMDB-a-4006.db"

engine = create_engine(f"sqlite:///{db_path}")

print("Database Connection established successfully!")

Database Connection established successfully!


## Database Relationships

The database follows a relational model where `movies` acts as the central entity.

### Direct Relationships

`movies`
- → `casts`
- → `genremap`
- → `keywordmap`
- → `languagemap`
- → `productioncompanymap`
- → `productioncountrymap`

### Dimension Relationships

`casts`
- → `actors`

`genremap`
- → `genres`

`keywordmap`
- → `keywords`

`languagemap`
- → `languages`

`productioncompanymap`
- → `productioncompanies`

`productioncountrymap`
- → `productioncountries`

### Relationship Pattern

movies
↓
bridge tables
↓
dimension tables

This structure creates several one-to-many and many-to-many relationships that must be handled carefully when constructing analytical datasets.

In [11]:
# Movies + Casts
query = """
SELECT
    m.movie_id,
    m.title,
    c.actor_id,
    c.characters
FROM movies m
INNER JOIN casts c
    ON m.movie_id = c.movie_id
LIMIT 20;
"""

movies_casts = pd.read_sql(query, engine)

movies_casts

,movie_id,title,actor_id,characters
0,5,Four Rooms,62,Leo
1,5,Four Rooms,138,Chester Rush
2,5,Four Rooms,2545,Long Hair Yuppy Scum
3,5,Four Rooms,2555,Siegfried
4,5,Four Rooms,3122,Jezebel
5,5,Four Rooms,3123,Diana
6,5,Four Rooms,3124,Athena
7,5,Four Rooms,3125,Elspeth
8,5,Four Rooms,3126,Eva
9,5,Four Rooms,3127,Raven


### INNER JOIN

An INNER JOIN returns only records where a matching key exists in both tables.

For `movies` and `casts`:

`movies.movie_id = casts.movie_id`

Therefore, only movies that have corresponding cast records are returned.

In [12]:
# LEFT JOIN
query = """
SELECT
    m.movie_id,
    m.title,
    c.actor_id,
    c.characters
FROM movies m
LEFT JOIN casts c
    ON m.movie_id = c.movie_id
LIMIT 20;
"""

movies_casts_left = pd.read_sql(query, engine)

movies_casts_left

,movie_id,title,actor_id,characters
0,5,Four Rooms,62,Leo
1,5,Four Rooms,138,Chester Rush
2,5,Four Rooms,2545,Long Hair Yuppy Scum
3,5,Four Rooms,2555,Siegfried
4,5,Four Rooms,3122,Jezebel
5,5,Four Rooms,3123,Diana
6,5,Four Rooms,3124,Athena
7,5,Four Rooms,3125,Elspeth
8,5,Four Rooms,3126,Eva
9,5,Four Rooms,3127,Raven


In [14]:
# movie → many cast records
pd.read_sql("""
SELECT
    m.movie_id,
    m.title,
    COUNT(c.actor_id) AS Actor_Count
FROM movies m
LEFT JOIN casts c
    ON m.movie_id = c.movie_id
GROUP BY
    m.movie_id,
    m.title
ORDER BY Actor_Count DESC;
""", engine).head(20)

,movie_id,title,Actor_Count
0,80585,Rock of Ages,224
1,3083,Mr. Smith Goes to Washington,213
2,82695,Les MisTrables,208
3,324668,Jason Bourne,208
4,10661,You Don't Mess with the Zohan,183
5,39254,Real Steel,172
6,13475,Star Trek,168
7,68728,Oz: The Great and Powerful,159
8,49026,The Dark Knight Rises,158
9,2976,Hairspray,152


In [15]:
# Movies ↔ Genres
query = """
SELECT
    m.movie_id,
    m.title,
    g.genre_id,
    g.genre_name
FROM movies m
INNER JOIN genremap gm
    ON m.movie_id = gm.movie_id
INNER JOIN genres g
    ON gm.genre_id = g.genre_id
LIMIT 30;
"""

movie_genres = pd.read_sql(query, engine)

movie_genres

,movie_id,title,genre_id,genre_name
0,11,Star Wars,12,Adventure
1,18,The Fifth Element,12,Adventure
2,22,Pirates of the Caribbean: The Curse of the Bla...,12,Adventure
3,58,Pirates of the Caribbean: Dead Man's Chest,12,Adventure
4,62,2001: A Space Odyssey,12,Adventure
5,74,War of the Worlds,12,Adventure
6,79,Hero,12,Adventure
7,85,Raiders of the Lost Ark,12,Adventure
8,87,Indiana Jones and the Temple of Doom,12,Adventure
9,89,Indiana Jones and the Last Crusade,12,Adventure


## Join Grain

Before joining tables, define the grain of each table.

### Example

`movies`

One row = one movie.

`casts`

One row = one actor/movie relationship.

`genremap`

One row = one movie/genre relationship.

`keywordmap`

One row = one movie/keyword relationship.

Therefore, joining `movies` to `casts` changes the resulting grain from:

One row = one movie

to:

One row = one movie/actor relationship.

This change in grain must be understood before calculating metrics.

In [ ]:
# Demonstrate Row Multiplication
query = """
SELECT
    m.movie_id,
    m.title,
    a.actor_name,
    g.genre_name,
    k.keyword_name,
    l.language_name,
    pc.production_company_name,
    pco.production_country_name
FROM movies m

LEFT JOIN casts c
    ON m.movie_id = c.movie_id

LEFT JOIN actors a
    ON c.actor_id = a.actor_id

LEFT JOIN genremap gm
    ON m.movie_id = gm.movie_id

LEFT JOIN genres g
    ON gm.genre_id = g.genre_id

LEFT JOIN keywordmap km
    ON m.movie_id = km.movie_id

LEFT JOIN keywords k
    ON km.keyword_id = k.keyword_id

LEFT JOIN languagemap lm
    ON m.movie_id = lm.movie_id

LEFT JOIN languages l
    ON lm.ISO_639_1 = l.ISO_639_1

LEFT JOIN productioncompanymap pcm
    ON m.movie_id = pcm.movie_id

LEFT JOIN productioncompanies pc
    ON pcm.production_company_id = pc.production_company_id

LEFT JOIN productioncountrymap pcom
    ON m.movie_id = pcom.movie_id

LEFT JOIN productioncountries pco
    ON pcom.ISO_3166_1 = pco.ISO_3166_1

WHERE m.movie_id = 19995;
"""

join_explosion = pd.read_sql(query, engine)

join_explosion.shape

(111552, 8)

In [18]:
join_explosion.head()

,movie_id,title,actor_name,genre_name,keyword_name,language_name,production_company_name,production_country_name
0,19995,Avatar,Giovanni Ribisi,Adventure,culture clash,English,Ingenious Film Partners,United States of America
1,19995,Avatar,Giovanni Ribisi,Adventure,culture clash,English,Ingenious Film Partners,United Kingdom
2,19995,Avatar,Giovanni Ribisi,Adventure,culture clash,English,Twentieth Century Fox Film Corporation,United States of America
3,19995,Avatar,Giovanni Ribisi,Adventure,culture clash,English,Twentieth Century Fox Film Corporation,United Kingdom
4,19995,Avatar,Giovanni Ribisi,Adventure,culture clash,English,Dune Entertainment,United States of America


In [19]:
# Measure the Multiplication
pd.read_sql("""
SELECT
    COUNT(*) AS Joined_Rows,
    COUNT(DISTINCT movie_id) AS Unique_Movies
FROM (
    SELECT
        m.movie_id
    FROM movies m

    LEFT JOIN genremap gm
        ON m.movie_id = gm.movie_id

    LEFT JOIN keywordmap km
        ON m.movie_id = km.movie_id

    LEFT JOIN languagemap lm
        ON m.movie_id = lm.movie_id

    LEFT JOIN productioncompanymap pcm
        ON m.movie_id = pcm.movie_id

    LEFT JOIN productioncountrymap pcom
        ON m.movie_id = pcom.movie_id
);
""", engine)

,Joined_Rows,Unique_Movies
0,974378,4803


In [21]:
# Correct Solution — Aggregate Before Joining
query = """
SELECT
    pcm.movie_id,
    GROUP_CONCAT(DISTINCT pc.production_company_name) AS Production_Companies
FROM productioncompanymap pcm
LEFT JOIN productioncompanies pc
    ON pcm.production_company_id = pc.production_company_id
GROUP BY pcm.movie_id;
"""

production_companies = pd.read_sql(query, engine)

production_companies.head()

,movie_id,Production_Companies
0,5,"Miramax Films,A Band Apart"
1,11,"Lucasfilm,Twentieth Century Fox Film Corporation"
2,12,Pixar Animation Studios
3,13,Paramount Pictures
4,14,"DreamWorks SKG,Jinks/Cohen Company"


In [23]:
query = """
SELECT
    gm.movie_id,
    GROUP_CONCAT(DISTINCT g.genre_name) AS Genres
FROM genremap gm
LEFT JOIN genres g
    ON gm.genre_id = g.genre_id
GROUP BY gm.movie_id;
"""

movie_genres = pd.read_sql(query, engine)
movie_genres.head()

,movie_id,Genres
0,5,"Comedy,Crime"
1,11,"Adventure,Action,Science Fiction"
2,12,"Animation,Family"
3,13,"Drama,Comedy,Romance"
4,14,Drama


In [24]:
query = """
SELECT
    lm.movie_id,
    GROUP_CONCAT(DISTINCT l.language_name) AS Languages
FROM languagemap lm
LEFT JOIN languages l
    ON lm.ISO_639_1 = l.ISO_639_1
GROUP BY lm.movie_id;
"""

movie_languages = pd.read_sql(query, engine)
movie_languages.head()

,movie_id,Languages
0,5,English
1,11,English
2,12,English
3,13,English
4,14,English


In [26]:
# Build a Clean Movie Master
query = """
SELECT
    m.movie_id,
    m.title,
    m.release_date,
    m.budget,
    m.revenue,
    m.popularity,

    g.Genres,
    l.Languages,
    pc.Production_Companies

FROM movies m

LEFT JOIN (
    SELECT
        gm.movie_id,
        GROUP_CONCAT(DISTINCT g.genre_name) AS Genres
    FROM genremap gm
    LEFT JOIN genres g
        ON gm.genre_id = g.genre_id
    GROUP BY gm.movie_id
) g
    ON m.movie_id = g.movie_id

LEFT JOIN (
    SELECT
        lm.movie_id,
        GROUP_CONCAT(DISTINCT l.language_name) AS Languages
    FROM languagemap lm
    LEFT JOIN languages l
        ON lm.ISO_639_1 = l.ISO_639_1
    GROUP BY lm.movie_id
) l
    ON m.movie_id = l.movie_id

LEFT JOIN (
    SELECT
        pcm.movie_id,
        GROUP_CONCAT(DISTINCT pc.production_company_name)
            AS Production_Companies
    FROM productioncompanymap pcm
    LEFT JOIN productioncompanies pc
        ON pcm.production_company_id = pc.production_company_id
    GROUP BY pcm.movie_id
) pc
    ON m.movie_id = pc.movie_id;
"""

movie_master = pd.read_sql(query, engine)

movie_master.head()

,movie_id,title,release_date,budget,revenue,popularity,Genres,Languages,Production_Companies
0,5,Four Rooms,1995-12-09 00:00:00.000000,4000000,4300000.0,22.876230,"Comedy,Crime",English,"Miramax Films,A Band Apart"
1,11,Star Wars,1977-05-25 00:00:00.000000,11000000,775398007.0,126.393695,"Adventure,Action,Science Fiction",English,"Lucasfilm,Twentieth Century Fox Film Corporation"
2,12,Finding Nemo,2003-05-30 00:00:00.000000,94000000,940335536.0,85.688789,"Animation,Family",English,Pixar Animation Studios
3,13,Forrest Gump,1994-07-06 00:00:00.000000,55000000,677945399.0,138.133331,"Drama,Comedy,Romance",English,Paramount Pictures
4,14,American Beauty,1999-09-15 00:00:00.000000,15000000,356296601.0,80.878605,Drama,English,"DreamWorks SKG,Jinks/Cohen Company"


In [27]:
movie_master.shape

(4803, 9)

In [28]:
movie_master["movie_id"].nunique()

4803

In [29]:
movie_master["movie_id"].duplicated().sum()

np.int64(0)